# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`
This notebook demonstrates how to load and analyze the FAIR^2 dataset using the `mlcroissant` library. The dataset is described with a Croissant schema that enables robust programmatic exploration of fields, record sets, and metadata.

### Dataset Source
The dataset source is provided via the Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
pd.set_option('display.max_columns', None)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset's Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets, fields, and associated `@id`s in the dataset.

In [ ]:
# List all record sets and their @id
print("Available record sets (by @id):")
record_sets = []
for rs in metadata.record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    record_sets.append(rs['@id'])

# For each record set, list its fields and the corresponding @id's
for rs in metadata.record_sets:
    print(f"\nRecord set: {rs.get('name', 'N/A')} (@id={rs['@id']})")
    print("  Fields:")
    for field in rs.get('fields', []):
        print(f"    - @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from one or several record sets by their `@id`s into Pandas DataFrames for analysis. All entity references use the exact `@id` string as identified above.

In [ ]:
# You can add or remove record set @id's here if the dataset defines multiple.
record_set_ids = record_sets

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Only load as DataFrame if records are tabular
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nFields (columns) for record set @id='{record_set_id}':")
        print(dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print(f"\nNo records found for record set @id='{record_set_id}'.")

## 4. Exploratory Data Analysis (EDA)
We apply standard tabular data processing and basic EDA. All operations reference columns and fields by their unique `@id`.

**Example Steps:**
- Select a numeric field (e.g., age or interval between diagnoses).
- Filter on a numeric threshold.
- Normalize the numeric column.
- Group by a categorical field (`@id`).

**Adjust `numeric_field_id` and `group_field_id` below to match the available `@id`s from Section 2.**

In [ ]:
# Choose one record set and relevant fields by @id for the EDA steps
# Please edit the following based on your data from above (e.g., age, sex, or other numeric/categorical identifiers)
# Here we assume an example @id for age: 'https://api.app.sen.science/frontiers/7862866/field-age'
# and for sex: 'https://api.app.sen.science/frontiers/7862866/field-sex'.
# Replace as needed with your actual field @ids.

if len(record_set_ids) > 0:
    rsid = record_set_ids[0]
    df = dataframes.get(rsid)
    if df is not None:
        # Print available columns
        print(f"Available columns for analysis (@id):\n{df.columns.tolist()}")
        
        # Try to identify a numeric_field @id (e.g., age) and group_field @id (e.g., sex or location)
        # For demonstration, we'll pick the first float/int column available, and another for grouping
        import numpy as np
        possible_numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if possible_numeric:
            numeric_field_id = possible_numeric[0]
            print(f"\nChosen numeric field for filtering/normalization: {numeric_field_id}")

            threshold = df[numeric_field_id].mean()  # use mean as example
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (mean):")
            display(filtered_df.head())

            # Normalize
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Pick a group field that's different from the numeric one
            possible_group = [c for c in df.columns if c != numeric_field_id and not pd.api.types.is_numeric_dtype(df[c])]
            if possible_group:
                group_field_id = possible_group[0]
                print(f"\nGrouping by field: {group_field_id}")
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
                print(f"Grouped means for {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
            else:
                print("No categorical field found for grouping.")
        else:
            print("No numeric field found in this record set.")
    else:
        print(f"No DataFrame found for record set {rsid}.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Use `matplotlib` or `seaborn` to visualize data distributions or variable relationships. Reference columns by their `@id` as in the above analyses.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field (if available)
if 'df' in locals() and df is not None and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If grouping field exists, show boxplot
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to programmatically load and analyze a Croissant-described dataset. Using unique `@id` fields, we loaded tabular data, performed filtering, normalization, grouping, and generated simple visualizations.

Remember, always refer to dataset documentation and Croissant schema for further details on fields and record sets for robust exploration.